In [ ]:
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("/content/Lending Club Data - DR_Demo_Lending_Club.csv")

features = ["annual_inc", "debt_to_income", "revol_util"]
target = "is_bad"

df_small = df[features + [target]].dropna()


scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_small[features].values)
y = df_small[target].values


i = 0
x = X_scaled[i]          # feature vector (x1, x2, x3)
y_true = float(y[i])     # true label (0 or 1)

print("Borrower index:", i)
print("Scaled features x:", x)
print("True label y_true:", y_true)

# ----------------------------------------
# 2. Initialize weights and bias
# ----------------------------------------
w = np.array([0.4, -0.3, 0.2], dtype=float)
b = -0.1

print("\nInitial parameters:")
print("w =", w)
print("b =", b)

# ----------------------------------------
# 3. Forward pass: compute z, p (sigmoid), and loss
# ----------------------------------------

# Linear Algebra - Matrix Math for multiple dimensions - Raw Logit
term1 = w[0] * x[0]
term2 = w[1] * x[1]
term3 = w[2] * x[2]
dot = term1 + term2 + term3
z = dot + b

print("\nLinear combination:")
print("term1 = w1*x1 =", term1)
print("term2 = w2*x2 =", term2)
print("term3 = w3*x3 =", term3)
print("dot = term1 + term2 + term3 =", dot)
print("z = dot + b =", z)

# Sigmoid function:
#
#           1
#   p = -----------
#        1 + e^(-z)


p = 1.0 / (1.0 + math.e**(-z))
e = math.e
exp_neg_z = e ** (-z)
p = 1.0 / (1.0 + exp_neg_z)
#
# This converts the linear score z into a probability p.

#         p
#         |
#      1  |                ______
#         |             __/
#         |          __/
#     0.5 |--------/
#         |      /
#         |   __/
#         |__/
#       0 +--------------------- z
#            -∞     0      +∞





print("\nSigmoid output:")
print("exp(-z) =", exp_neg_z)
print("p = sigmoid(z) =", p, "  # P(loan is bad | x)")




# # Logistic (binary cross-entropy) loss for one sample:
#
#   L = -[ y * log(p) + (1 - y) * log(1 - p) ]
#
# We split it into two cases for numerical stability:
#   - If y = 1:  L = -log(p)
#   - If y = 0:  L = -log(1 - p)
#
# We add a tiny epsilon (1e-9) inside the log to avoid log(0),
# which would cause math errors when p is extremely close to 0 or 1.
#
# eps prevents issues like:
#   log(0)       → undefined
#   log(1 - 1)   → undefined
#
# Binary cross-entropy loss for this one borrower
eps = 1e-9  # avoid log(0)

if y_true == 1:
    loss = -math.log(p + eps)
else:
    loss = -math.log(1.0 - p + eps)

print("\nLogistic loss for this borrower:")
print("L =", loss)



# ----------------------------------------
# 4. Backward pass, step-by-step: derivatives
# ----------------------------------------


# Differentiate L with respect to p:
#  d/dp [-y log(p)]             = -y / p
#  d/dp [-(1-y) log(1-p)]       = (1 - y) / (1 - p)
#
# So the full derivative is:
#       dL/dp = -y/p + (1 - y)/(1 - p)
#
# We add eps to avoid division by zero when p is extremely close to 0 or 1.

# How much does the loss change when the predicted probability p changes just a tiny bit?
eps = 1e-9  # avoid log(0)
if y_true == 1:
    loss = -math.log(p + eps)
else:
    loss = -math.log(1.0 - p + eps)

print("\nLogistic loss for this borrower:")
print("L =", loss)


# --- dL/dp: derivative of BCE wrt probability p ---
# L = -[y log(p) + (1-y) log(1-p)]
term1 = -y_true / (p + eps)             # derivative of -y log(p)
term2 = (1.0 - y_true) / (1.0 - p + eps)  # derivative of -(1-y) log(1-p)
dL_dp_value = term1 + term2

print("\nDerivative wrt p:")
print("term1 = -y/p =", term1)
print("term2 = (1-y)/(1-p) =", term2)
print("dL/dp =", dL_dp_value)

# --- dp/dz: derivative of sigmoid ---
# p = sigmoid(z), dp/dz = p(1-p)
dp_dz = p * (1.0 - p)
print("\nDerivative of sigmoid:")
print("dp/dz =", dp_dz)

# --- dL/dz via chain rule ---
# dL/dz = (dL/dp) * (dp/dz)
dL_dz_chain = dL_dp_value * dp_dz
print("\ndL/dz (from chain rule) =", dL_dz_chain)

# Simplified form: dL/dz = p - y
dL_dz_simplified = p - y_true
print("dL/dz (simplified p - y) =", dL_dz_simplified)

# We'll use the simplified one going forward
dL_dz = dL_dz_simplified
print("\nUsing dL/dz =", dL_dz)

# ----------------------------------------
# 5. Derivatives of z wrt weights and bias
# ----------------------------------------
# z = w1*x1 + w2*x2 + w3*x3 + b
dz_dw1 = x[0]
dz_dw2 = x[1]
dz_dw3 = x[2]
dz_db = 1.0

print("\nDerivatives of z wrt weights and bias:")
print("dz/dw1 =", dz_dw1)
print("dz/dw2 =", dz_dw2)
print("dz/dw3 =", dz_dw3)
print("dz/db  =", dz_db)

# ----------------------------------------
# 6. Gradients wrt weights and bias (chain rule)
# ----------------------------------------
# dL/dw_j = dL/dz * dz/dw_j = (p - y) * x_j
dL_dw1 = dL_dz * dz_dw1
dL_dw2 = dL_dz * dz_dw2
dL_dw3 = dL_dz * dz_dw3

print("\nGradients wrt each weight:")
print("dL/dw1 =", dL_dw1)
print("dL/dw2 =", dL_dw2)
print("dL/dw3 =", dL_dw3)

# Vector gradient for all weights
grad_w = dL_dz * x
print("grad_w (vector) =", grad_w)

# Bias gradient: dL/db = dL/dz * dz/db = p - y
dL_db = dL_dz * dz_db
print("\nGradient wrt bias:")
print("dL/db =", dL_db)

# ----------------------------------------
# 7. One gradient descent update step
# ----------------------------------------
learning_rate = 0.1

w_new = w - learning_rate * grad_w
b_new = b - learning_rate * dL_db

print("\nUpdated parameters after one GD step:")
print("w_new =", w_new)
print("b_new =", b_new)

# ----------------------------------------
# 8. Recompute prediction with updated parameters
# ----------------------------------------
term1_new = w_new[0] * x[0]
term2_new = w_new[1] * x[1]
term3_new = w_new[2] * x[2]
dot_new = term1_new + term2_new + term3_new
z_new = dot_new + b_new

exp_neg_z_new = e ** (-z_new)
p_new = 1.0 / (1.0 + exp_neg_z_new)

if y_true == 1:
    loss_new = -math.log(p_new + eps)
else:
    loss_new = -math.log(1.0 - p_new + eps)

print("\nAfter update: forward pass again")
print("z_new =", z_new)
print("p_new = sigmoid(z_new) =", p_new)
print("New loss L_new =", loss_new)
print("\nChange in loss:", loss_new - loss, "(should be <= 0 for a good step)")




Borrower index: 0
Scaled features x: [-0.3758329  -0.36711907 -1.28816001]
True label y_true: 0.0

Initial parameters:
w = [ 0.4 -0.3  0.2]
b = -0.1

Linear combination:
term1 = w1*x1 = -0.1503331617617274
term2 = w2*x2 = 0.1101357196985946
term3 = w3*x3 = -0.2576320029012013
dot = term1 + term2 + term3 = -0.2978294449643341
z = dot + b = -0.39782944496433414

Sigmoid output:
exp(-z) = 1.488590121713932
p = sigmoid(z) = 0.4018339505869629   # P(loan is bad | x)

Logistic loss for this borrower:
L = 0.5138868873004455

Derivative wrt p:
term1 = -y/p = -0.0
term2 = (1-y)/(1-p) = 1.6717765899778068
dL/dp = 1.6717765899778068

Derivative of sigmoid:
dp/dz = 0.24036342674263717

dL/dz (from chain rule) = 0.40183394991518634
dL/dz (simplified p - y) = 0.4018339505869629

Using dL/dz = 0.4018339505869629

Derivatives of z wrt weights and bias:
dz/dw1 = -0.3758329044043185
dz/dw2 = -0.36711906566198205
dz/dw3 = -1.2881600145060066
dz/db  = 1.0

Gradients wrt each weight:
dL/dw1 = -0.1510224207